In [57]:
import matplotlib.pyplot as plt
import numpy as np
import pyarrow as pa
import kete

Going through the __[kete Getting Started](https://dahlend.github.io/kete/auto_examples/getting_started.html)__ article:

States:

In [2]:
# Lets construct an object circular orbit
# Recall that an object in circular orbit has a speed of sqrt(GM / r) where r is the
# radius of the orbit.
r = 1
epoch = kete.Time.j2000()
speed = np.sqrt(kete.constants.SUN_GM / r)
pos = kete.Vector([0, r, 0], frame=kete.Frames.Equatorial)
vel = kete.Vector([-speed, 0, 0], frame=kete.Frames.Equatorial)

# We now make a State, representing the full information for an object in orbit of the
# sun.
state = kete.State("Circular", epoch, pos, vel)
state


State(desig="Circular", jd=2451545.0, pos=[0.0, 1.0, 0.0], vel=[-0.017202098949957226, 0.0, 0.0], frame=Equatorial, center="sun")

In [3]:
# Grabbing the state of the Moon.
# kete includes spice kernels for all of the planets, the Moon, Pluto and 5 asteroids:
# Ceres, Pallas, Vesta, Hygiea, Interamnia
jd = kete.Time.from_ymd(2020, 1, 1).jd
moon = kete.spice.get_state("Moon", jd)

# Grabbing a state from JPL Horizons
# Note that the epoch of this state is the epoch of fit from Horizons.
eros = kete.HorizonsProperties.fetch("Eros").state

# Grabbing a state from the MPC
# This is more expensive, as the MPC lookup involves downloading a full copy of the MPC
# database. This database contains more than a million asteroids. This gets returned as
# a pandas dataframe.
orb_file = kete.mpc.fetch_known_orbit_data()
subset = orb_file[orb_file["name"] == "Eros"]

# Note that in this case the tool converts all objects in the provided table into states
# which means it will always return a list of states, so below we get a list containing
# a single state.
states = kete.conversion.table_to_states(subset)
states

[State(desig="433", jd=2461200.500800741, pos=[-1.1589938927476588, -0.6900560046811977, -0.2575260481280023], vel=[0.004824360587328715, -0.014288854137296986, -0.0007763205349132643], frame=Ecliptic, center="sun")]

Propagation:

In [4]:
# Lets grab the state of a few asteroids:
names = ["Eros", "Bennu", "Ryugu"]

orb_file = kete.mpc.fetch_known_orbit_data()
subset = orb_file[[name in names for name in orb_file["name"]]]

# This now contains 3 states:
states = kete.conversion.table_to_states(subset)

# Lets propagate these states to the current epoch
now = kete.Time.now().jd

states_now_exact = kete.propagate_n_body(states, now)
states_now_kepler = kete.propagate_two_body(states, now)
states_now_exact

SimultaneousStates(states=<3 States>, fov=None)

In [12]:
len(states_now_exact)

3

Field of Views

In [4]:
now = kete.Time.now().jd

# lookup the observers position in the solar system from a given observatory code/name.
observer_a = kete.spice.mpc_code_to_ecliptic("Palomar Mountain", now)

# lookup the observer position from an earth location, this is IPAC, at 100m altitude.
observer_b = kete.spice.earth_pos_to_ecliptic(now, 33.133278, 241.87206, 0.1, "IPAC")
observer_b

State(desig="IPAC", jd=2461252.399015301, pos=[0.6160502476883779, -0.806733795232629, 6.0126287585060556e-5], vel=[0.013289521830155656, 0.010187142382644407, 7.90100188751185e-5], frame=Ecliptic, center="sun")

In [5]:
fov = kete.OmniDirectionalFOV(observer_a)
fov.observer.pos

Vector([0.616049975809, -0.806734421024, 6.0554041e-5], Ecliptic)

In [8]:
anchors = np.asarray(fov.observer.pos)

In [12]:
results = kete.fov_state_check(states_now_exact,[fov,fov])

In [ ]:
for p,sim in enumerate(results): # does however many fovs were in the fov array of fov_state_check()
    print(p,sim,sim.ra_dec_with_rates)


0 SimultaneousStates(states=<3 States>, fov=OmniDirectional(observer=State(desig="675", jd=2461246.490416285, pos=[0.5340330071089889, -0.8639756038950155, 7.353512157398656e-5], vel=[0.014356005088817028, 0.00877758782911802, 8.857543705617253e-5], frame=Ecliptic, center="sun"))) [[190.44152416494674, -16.059109145259477, 0.6429273115385763, -0.1852297140460944], [165.32683498116054, 4.9694408232634455, 0.7429239731579701, -0.36211433947121896], [177.56077372706375, -1.694206680503057, 0.7906156369344195, -0.2867174618986267]]
1 SimultaneousStates(states=<3 States>, fov=OmniDirectional(observer=State(desig="675", jd=2461246.490416285, pos=[0.5340330071089889, -0.8639756038950155, 7.353512157398656e-5], vel=[0.014356005088817028, 0.00877758782911802, 8.857543705617253e-5], frame=Ecliptic, center="sun"))) [[190.44152416494674, -16.059109145259477, 0.6429273115385763, -0.1852297140460944], [165.32683498116054, 4.9694408232634455, 0.7429239731579701, -0.36211433947121896], [177.5607737270

Figuring out arrays from positions and quantities with `hdps.catalogs.movers`

In [3]:
desigs = [1,100,1000]

orb_file = kete.mpc.fetch_known_orbit_data()
subset = orb_file[[name in desigs for name in orb_file["desig"]]]

# This now contains 3 states:
states = kete.conversion.table_to_states(subset)

# Lets propagate these states to the current epoch
now = kete.Time.now().jd

states_now_exact = kete.propagate_n_body(states, now)
states_now_kepler = kete.propagate_two_body(states, now)
states_now_exact[0].pos # there should be 3

Vector([0.912886735212, 2.563631913187, -0.087013491322], Ecliptic)

In [ ]:
now = kete.Time.now().jd
tomorrow = kete.Time.now().jd+1
# epoch list
anchors = [now,now+1/24/60,now+2/24/60,now+3/24/60,now+4/24/60]
# lookup the observers position in the solar system from a given observatory code/name.
observers = [kete.spice.mpc_code_to_ecliptic("Palomar Mountain", i) for i in anchors]
#observer_b = kete.spice.mpc_code_to_ecliptic("Palomar Mountain", tomorrow)
fovs = [kete.OmniDirectionalFOV(i) for i in observers]
#fov_b = kete.OmniDirectionalFOV(observer_b)
#fov_a.observer.pos
#results = kete.fov_state_check(states_now_exact,fovs) #[fov_a,fov_b])

In [8]:
# go through orbits.load_and_propagate_states()
# get orbital element file
# done in orbits._fetch_orbit_table -> services.mpc.fetch_mpc_orbit_table()
orb_file = kete.mpc.fetch_known_orbit_data() 
# get states from orbital element file
# done in orbits._propagate_states()
initial_states = kete.conversion.table_to_states(orb_file) 
# propagate states to now with an n-body propagation 
# done in orbits._propagate_states()
propagated = kete.propagation.propagate_n_body(initial_states,now) 
# returns 2 SimultaneousStates(), one for each fov.  
# done in ephemerides._probe_night()
results = kete.fov_state_check(list(propagated),fovs) # objects in given states are propagated to fov epoch if less than dt_limit (default = 3)

Impact detected between (2008 TC3) <-> earth at time 2454746.6142098513 (2008-10-07T02:43:22.547+00:00)
Impact detected between (2014 AA) <-> earth at time 2456659.602368593 (2014-01-02T02:26:17.462+00:00)
Impact detected between (2022 EB5) <-> earth at time 2459650.3938325476 (2022-03-11T21:25:57.948+00:00)
Impact detected between (2022 WJ1) <-> earth at time 2459902.8530103927 (2022-11-19T08:27:10.913+00:00)
Impact detected between (2023 CX1) <-> earth at time 2459988.626554247 (2023-02-13T03:01:05.102+00:00)
Impact detected between (2024 BX1) <-> earth at time 2460330.5247700023 (2024-01-21T00:34:30.944+00:00)
Impact detected between (2024 RW1) <-> earth at time 2460558.195744491 (2024-09-04T16:40:43.140+00:00)
Impact detected between (2024 UQ) <-> earth at time 2460605.955892028 (2024-10-22T10:55:19.887+00:00)
Impact detected between (2024 XA1) <-> earth at time 2460648.178989184 (2024-12-03T16:16:35.481+00:00)
Impact detected between (2026 JN4) <-> earth at time 2461176.0716617624

In [9]:
# to populate with r, delta, alpha, V
samples = np.full((4,len(anchors),len(results[0])),0.0)
for p,sim in enumerate(results):
    obs_vecs = sim.obs_vecs # obs2obj, all state observer to object vectors for first fov
    sun2obs = np.asarray(fovs[p].observer.pos) # (3,) sun to observer vector for first fov
    helio_pos = sun2obs[None,:]+obs_vecs # (N_obj,3) sun2obj
    obj2obs = sun2obs - helio_pos # 
    r = np.linalg.norm(helio_pos,axis=1) # (N_obj,)
    delta = np.linalg.norm(obj2obs,axis=1) # (N_obj,)
    samples[0,p,:] = r
    samples[1,p,:] = delta
samples

array([[[2.72272313, 2.9578703 , 2.7876364 , ..., 3.34115103,
         1.91061651, 2.6777062 ],
        [2.72272256, 2.95786882, 2.78763464, ..., 3.34115304,
         1.91061619, 2.67770475],
        [2.72272199, 2.95786733, 2.78763288, ..., 3.34115504,
         1.91061586, 2.6777033 ],
        [2.72272142, 2.95786585, 2.78763111, ..., 3.34115705,
         1.91061554, 2.67770185],
        [2.72272084, 2.95786437, 2.78762935, ..., 3.34115906,
         1.91061522, 2.6777004 ]],

       [[3.37659677, 2.51020904, 1.79687378, ..., 2.35857283,
         2.47718964, 2.25501708],
        [3.37658988, 2.5101984 , 1.79687307, ..., 2.35857706,
         2.47719357, 2.25500679],
        [3.37658298, 2.51018776, 1.79687237, ..., 2.35858128,
         2.47719749, 2.25499651],
        [3.37657609, 2.51017712, 1.79687166, ..., 2.35858551,
         2.47720141, 2.25498623],
        [3.37656919, 2.51016648, 1.79687096, ..., 2.35858973,
         2.47720534, 2.25497594]],

       [[0.        , 0.        , 0. 

In [11]:
q,t,n = samples.shape
print(q,t,n)

4 5 1553127


In [14]:
samp = samples[1]
p,n = samp.shape #(p anchors, n objects)
finite = np.isfinite(samp)
cnt = finite.sum(axis=0)
full = cnt == p
full

array([ True,  True,  True, ...,  True,  True,  True], shape=(1553127,))

In [ ]:
x = [-1,-0.1,0,0.1,1]
out = np.zeros((4,n))
# generates 4 coefficients for each object
out[:,full] = np.polynomial.chebyshev.chebfit(x,samp[:,full],3)  

In [ ]:
# build the dictionary in records._NightlyChebyshevModel.with_codes()
desigs = np.array([str(i) for i in orb_file.desig])
order = np.argsort(desigs,kind="stable")
dictionary = desigs[order]
codes = np.empty(desigs.shape[0],dtype=np.int32)
codes[order] = np.arange(desigs.shape[0],dtype=np.int32)
dictionary[codes] # gives you desigs in the order in which they appear 

In [64]:
dictionary

array(['1', '10', '100', ..., '99997', '99998', '99999'],
      shape=(1553137,), dtype='<U10')

In [66]:
coeffs = np.zeros((n,4,4))
designation = pa.DictionaryArray.from_arrays(
    pa.array(np.arange(coeffs.shape[0],dtype=np.int32)),
    pa.array(dictionary,type=pa.large_string())
    )

In [68]:
cols: dict[str,"pa.Array"] = {"designation":designation}

In [69]:
cols

{'designation': <pyarrow.lib.DictionaryArray object at 0x7f23ed11a7a0>
 
 -- dictionary:
   [
     "1",
     "10",
     "100",
     "1000",
     "10000",
     "100000",
     "100001",
     "100002",
     "100003",
     "100004",
     ...
     "99990",
     "99991",
     "99992",
     "99993",
     "99994",
     "99995",
     "99996",
     "99997",
     "99998",
     "99999"
   ]
 -- indices:
   [
     0,
     1,
     2,
     3,
     4,
     5,
     6,
     7,
     8,
     9,
     ...
     1553117,
     1553118,
     1553119,
     1553120,
     1553121,
     1553122,
     1553123,
     1553124,
     1553125,
     1553126
   ]}

In [71]:
images = ["1","2","3","4"]
for idx,image in enumerate(images):
    if image == "3":
        print("we're at 3")
        continue
    print("im = ",image)

im =  1
im =  2
we're at 3
im =  4
